# Retail Sales Data Analysis & Business Insights
**Track:** Data Analytics | **Level 1:** Exploratory Data Analysis
**Tools:** Python, pandas, matplotlib, seaborn


### Dataset Description ###

This project analyses an electronics retail sales dataset covering transactions from
September 2023 to September 2024. Each row represents one transaction.

**Numerical:** Age, Rating, Unit Price, Quantity, Total Price
**Categorical:** Gender, Loyalty Member, Product Type, SKU, Order Status, Payment Method, Shipping Type
**Date:** Purchase Date
**Multi-valued text:** Add-ons Purchased

**Adaptations:** The dataset has no separate "product category" column, so
`Product Type` is used as the category and `SKU` as the product identifier.
`Total Price` is used as revenue.

In [42]:
import pandas as pd

pd.read_csv("Electronic_sales_Sep2023-Sep2024.csv")
df.head()
df.columns.tolist()


['Customer ID',
 'Age',
 'Gender',
 'Loyalty Member',
 'Product Type',
 'SKU',
 'Rating',
 'Order Status',
 'Payment Method',
 'Total Price',
 'Unit Price',
 'Quantity',
 'Purchase Date',
 'Shipping Type',
 'Add-ons Purchased',
 'Add-on Total']

## 1. Business Problem / Objective
An electronics retailer wants to understand its sales performance. This analysis aims to answer:
- How do sales change over time (monthly and quarterly)?
- Who are the customers (age group, gender)?
- Which products and categories generate the most revenue?
- Which numerical variables are related?
- What actionable steps can the business take?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")   # clean default chart look
plt.rcParams["figure.figsize"] = (10, 5)  # default chart size
df = pd.read_csv(r"D:\My Downloads\Electronic_sales_Sep2023-Sep2024.csv")
df.head()

# Initial Data Inspection #

### Observation
- The dataset contains 20,000 transactions and 16 columns, with no fully duplicated rows.
- `Gender` has 1 missing value (0.005%) and `Add-ons Purchased` has 4,868 missing values (24.34%); all other columns are complete.
- `Purchase Date` is stored as text rather than a date type.
- `Payment Method` contains both "Paypal" and "PayPal", an inconsistent label for the same method.
- Numeric ranges are plausible: Age 18-80, Rating 1-5, Quantity 1-10, Unit Price 20.75-1,139.68.


In [ ]:
# 1. Size
df.shape

In [ ]:
# 2. Last rows (checks the file wasn't cut off)
df.tail()

In [ ]:
# 3. Column names, data types, non-null counts
df.info()

### Insight
The data is largely complete, but three quality issues must be fixed before analysis:
the date type, the inconsistent PayPal label, and the missing add-on values (likely "no add-ons",
as Add-on Total is 0.00 in the rows inspected). Order Status includes both "Completed" and
"Cancelled", so we must decide how cancelled orders affect revenue.


In [ ]:
# 4. Missing values: count and percentage
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "percent": missing_pct})

In [ ]:
# 5. Fully duplicated rows
df.duplicated().sum()

In [ ]:
# 6. Unique values in categorical columns
for col in ["Gender", "Loyalty Member", "Product Type", "Order Status", "Payment Method", "Shipping Type"]:
    print(col, "->", df[col].unique())

In [ ]:
# 7. Quick numeric sanity check (min/max reveal impossible values)
df[["Age", "Rating", "Unit Price", "Quantity", "Total Price"]].describe()

In [ ]:
# 1. How many orders are cancelled?
df["Order Status"].value_counts()

In [ ]:
# 2. Does Total Price = Unit Price x Quantity in every row?
price_diff = (df["Unit Price"] * df["Quantity"] - df["Total Price"]).abs()
print("Rows that don't match:", (price_diff > 0.01).sum())
print("Largest difference:", price_diff.max())

In [ ]:
# 3. Do missing add-ons really mean "no add-ons"?
no_addon_rows = df[df["Add-ons Purchased"].isnull()]
print("Add-on Total when add-ons are missing:")
print(no_addon_rows["Add-on Total"].describe())

In [ ]:
# 4. SKU check: how many are there, and what formats?
print("Unique SKUs:", df["SKU"].nunique())
print(df.groupby("Product Type")["SKU"].unique())

In [ ]:
# 5. Can every date be converted? (errors="coerce" turns bad dates into NaT instead of crashing)
test_dates = pd.to_datetime(df["Purchase Date"], errors="coerce")
print("Unconvertible dates:", test_dates.isnull().sum())
print("Date range:", test_dates.min(), "to", test_dates.max())

In [ ]:
# 6. The single missing Gender row, and the PayPal labels
print(df[df["Gender"].isnull()])
print(df["Payment Method"].value_counts())

In [ ]:
df_clean = df.copy()   
df_clean["Purchase Date"] = pd.to_datetime(df_clean["Purchase Date"])
df_clean["Payment Method"] = df_clean["Payment Method"].replace({"Paypal": "PayPal"})
df_clean["Gender"] = df_clean["Gender"].fillna("Unknown")
df_clean["Add-ons Purchased"] = df_clean["Add-ons Purchased"].fillna("None")
df_clean["Product"] = df_clean["Product Type"] + " (" + df_clean["SKU"] + ")"

# Revenue analysis ke liye sirf completed orders
df_completed = df_clean[df_clean["Order Status"] == "Completed"].copy()

In [ ]:
# Verification
print("Date type:", df_clean["Purchase Date"].dtype)
print("Total nulls left:", df_clean.isnull().sum().sum())
print(df_clean["Payment Method"].value_counts())
print("All rows:", len(df_clean), "| Completed rows:", len(df_completed))
print("Unique products:", df_clean["Product"].nunique())

### Data Cleaning Decisions

| Issue | Why it matters | Action |
|---|---|---|
| 32.84% of orders (6,568) are Cancelled | Including them would overstate revenue | Revenue analysis uses Completed orders only; cancelled rows retained for cancellation analysis |
| Purchase Date stored as text | Time-based analysis needs a datetime type | Converted with pd.to_datetime (0 invalid dates) |
| "Paypal" and "PayPal" both present | Same method counted as two categories | Standardised to "PayPal" |
| 1 missing Gender value | Deleting the row would discard a valid transaction | Filled with "Unknown" |
| 4,868 missing Add-ons Purchased | Add-on Total is 0.00 for all of them, so missing means no add-ons | Filled with "None" |
| One SKU appears under multiple Product Types | SKU alone would merge different products | Created `Product` = Product Type + SKU |
| Total Price = Unit Price × Quantity in all rows | Confirms revenue column is reliable | No change needed |
| Highest Total Price (11,396.80) equals max Unit Price × max Quantity | Indicates genuine large orders, not errors | Outliers retained |
| Data spans 2023-09-24 to 2024-09-23 | First and last months are partial | Considered in time-series interpretation |
| 10 distinct SKU codes map to 13 Product Type + SKU combinations | A SKU code alone is ambiguous across product types | Analysis uses the combined `Product` column |

In [ ]:
num_cols = ["Age", "Rating", "Unit Price", "Quantity", "Total Price", "Add-on Total"]

stats = df_completed[num_cols].describe().T
stats["median"] = df_completed[num_cols].median()
stats["mode"] = df_completed[num_cols].mode().iloc[0]
stats["skew"] = df_completed[num_cols].skew()

stats = stats[["count", "mean", "median", "mode", "std", "min", "25%", "75%", "max", "skew"]].round(2)
stats

### Observation
- Total Price is moderately right-skewed (skew 0.90): the mean (3,173.74) is about 25% above the median (2,534.49), and the standard deviation (2,541.48) is roughly 80% of the mean.
- Add-on Total is also right-skewed (0.94) with a median of 52.11 and a mode of 0.00, so the most common case is an order with no add-ons.
- Age (mean 48.97, median 49), Quantity (mean 5.48, median 5), Rating (mean 3.10, median 3) and Unit Price show skew close to zero, so they are roughly symmetric.
- The middle 50% of customers are aged 33 to 65.

### Insight
Order values vary widely and a minority of large orders pulls the average upward, so the median is a better indicator of a typical order than the mean. Add-ons are small compared with order value (average 62.21 vs 3,173.74) and many orders include none, which suggests add-on sales are under-used.

### Action
Report median alongside mean when discussing order value, and consider segmenting customers by order size. The business could test add-on bundles at checkout, since a large share of orders currently include no add-ons.

In [ ]:
# Outlier check for Total Price using the IQR rule
q1 = df_completed["Total Price"].quantile(0.25)
q3 = df_completed["Total Price"].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr

high_orders = df_completed[df_completed["Total Price"] > upper_limit]
print("Upper limit:", round(upper_limit, 2))
print("Orders above limit:", len(high_orders))

In [ ]:
# Is Total Price skewed?
sns.histplot(df_completed["Total Price"], bins=30, kde=True)
plt.title("Distribution of Total Price (Completed Orders)")
plt.xlabel("Total Price")
plt.ylabel("Number of Orders")
plt.show()

### Observation
- Total Price is right-skewed: the most common order value is the lowest range (roughly 0-400, about 1,600 orders), and the number of orders declines steadily toward a long tail reaching about 11,400.
- Using the IQR rule, 262 completed orders (about 1.95% of 13,432) exceed the upper limit of 9,973.69.
- These high-value orders are consistent with the maximum possible combination of unit price and quantity (11,396.80), so they are genuine large purchases rather than data errors.

### Insight
Most transactions are small, while a small group of large orders stretches the distribution to the right. Because of this skew, the average order value can be pulled upward by these large orders, so the median is a more reliable measure of a "typical" order.

### Action
Keep the high-value orders in the analysis, and report both mean and median when discussing order value. The business could track these large orders separately, since a small number of them may contribute a disproportionate share of revenue.

In [ ]:
# Date se naye columns
df_completed["Year"] = df_completed["Purchase Date"].dt.year
df_completed["Month"] = df_completed["Purchase Date"].dt.month
df_completed["Month Name"] = df_completed["Purchase Date"].dt.month_name()
df_completed["Quarter"] = df_completed["Purchase Date"].dt.quarter
df_completed["Year-Month"] = df_completed["Purchase Date"].dt.to_period("M")

# Monthly revenue aur orders
monthly = df_completed.groupby("Year-Month").agg(
    revenue=("Total Price", "sum"),
    orders=("Total Price", "count")
)
monthly
monthly["avg_order_value"] = (monthly["revenue"] / monthly["orders"]).round(2)
monthly

In [ ]:
# Monthly Sales Trend
months = monthly.index.astype(str)
revenue_m = monthly["revenue"] / 1_000_000   # millions mein

plt.figure(figsize=(12, 5))
plt.plot(months, revenue_m, marker="o", label="Monthly revenue")
plt.scatter([months[0], months[-1]], [revenue_m.iloc[0], revenue_m.iloc[-1]],
            color="red", zorder=5, label="Partial month")
plt.title("Monthly Revenue Trend (Completed Orders)")
plt.xlabel("Month")
plt.ylabel("Revenue (millions)")
plt.xticks(rotation=45)
plt.legend()
plt.show()

### Observation
- Orders rose from 525 in December 2023 to 1,399 in January 2024 (+166%), while average order value rose from 2,495 to 3,228 (+29%), lifting revenue about 3.4 times (1.31M to 4.52M).
- From January to August 2024, orders stayed between 1,227 and 1,399 per month and average order value between 3,176 and 3,349; January was the highest-revenue month and February the lowest.
- September 2024 (up to the 23rd) had about 43.5 orders per day versus about 42.7 in August, so its lower monthly total reflects fewer days, not weaker sales.

### Insight
The January 2024 jump was driven mainly by order volume (about 2.7x) and partly by higher order value (about 1.3x). After the jump, both measures stayed stable, which suggests a lasting change in business conditions rather than a seasonal spike. The data cannot show the cause, and since the dataset is synthetic the step may reflect how it was generated.

### Action
Investigate what changed around January 2024 (campaigns, product launches, channels). Use 2024 levels (about 1,300 orders and 3,300 average order value per month) as the baseline for targets, and judge partial months on a per-day basis.

In [ ]:
# Quarterly Sales Trend (2023 Q3 hata rahe hain, sirf 7 din ka hai)
quarterly = df_completed.groupby(["Year", "Quarter"])["Total Price"].sum()
quarterly = quarterly.drop((2023, 3))
labels = [f"{y} Q{q}" for y, q in quarterly.index]

plt.figure(figsize=(8, 5))
plt.plot(labels, quarterly.values / 1_000_000, marker="o")
plt.title("Quarterly Revenue Trend (Completed Orders)")
plt.xlabel("Quarter")
plt.ylabel("Revenue (millions)")
plt.ylim(0, None)
plt.show()
quarterly

### Observation
- Revenue was 4.26M in 2023 Q4, 12.64M in 2024 Q1 (about +197%), 13.23M in Q2 (+4.7%) and 12.22M in Q3 (-7.6%).
- Q3 2024 contains only 85 days of data. Per day, it is about 143.8K versus 145.4K in Q2, a difference of about 1.1%.

### Insight
Nearly all growth happened between 2023 Q4 and 2024 Q1, after which quarterly revenue is flat at roughly 12-13M. The apparent Q3 decline is mostly caused by missing days, not weaker performance.

### Action
Compare quarters on revenue per day and treat 2024 Q1-Q2 as the baseline until a full Q3 is available.

In [ ]:
age_bins = [17, 29, 44, 59, 80]
age_labels = ["Young Adult (18-29)", "Adult (30-44)", "Middle-aged (45-59)", "Senior (60+)"]
df_completed["Age Group"] = pd.cut(df_completed["Age"], bins=age_bins, labels=age_labels)

age_summary = df_completed.groupby("Age Group", observed=True).agg(
    orders=("Total Price", "count"),
    revenue=("Total Price", "sum"),
    avg_order_value=("Total Price", "mean")
).round(2)
age_summary["revenue_share_%"] = (age_summary["revenue"] / age_summary["revenue"].sum() * 100).round(1)
age_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_data = age_summary.reset_index()

sns.barplot(data=plot_data, x="Age Group", y="orders", color="steelblue", ax=axes[0])
axes[0].set_title("Completed Orders by Age Group")
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("Number of Orders")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=plot_data, x="Age Group", y="avg_order_value", color="seagreen", ax=axes[1])
axes[1].set_title("Average Order Value by Age Group")
axes[1].set_xlabel("Age Group")
axes[1].set_ylabel("Average Order Value")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
group_width = {"Young Adult (18-29)": 12, "Adult (30-44)": 15, "Middle-aged (45-59)": 15, "Senior (60+)": 21}
age_summary["orders_per_age_year"] = (age_summary["orders"] / pd.Series(group_width)).round(1)
age_summary[["orders", "orders_per_age_year"]]

### Observation
- The Senior group (60+) has the most completed orders (4,469) and the largest revenue share (33.3%), followed by Adult 30-44 (3,295 orders, 24.3%), Middle-aged 45-59 (3,142, 23.5%) and Young Adult 18-29 (2,526, 18.9%).
- Average order value is almost identical across groups, ranging only from 3,143 to 3,194 (about 1.6% difference).
- The Senior group spans 21 years of age versus 12-15 years for the others. Per year of age, orders are similar in every group (roughly 209-220).

### Insight
Differences in revenue between age groups come from the number of orders, not from how much each customer spends per order. The Senior group's lead is largely explained by its wider age range, so no age group stands out as clearly more valuable once group size is accounted for.

### Action
Age alone is not a strong lever for raising order value, so avoid age-based pricing assumptions. Keep marketing broad across age groups, and look to other segments (loyalty, product, payment method) for differences that could guide targeting.

In [ ]:
gender_summary = df_completed.groupby("Gender").agg(
    orders=("Total Price", "count"),
    revenue=("Total Price", "sum"),
    avg_order_value=("Total Price", "mean")
).round(2)
gender_summary["revenue_share_%"] = (gender_summary["revenue"] / gender_summary["revenue"].sum() * 100).round(1)
gender_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_gender = gender_summary.drop("Unknown").reset_index()

sns.barplot(data=plot_gender, x="Gender", y="orders", color="steelblue", ax=axes[0])
axes[0].set_title("Completed Orders by Gender")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Number of Orders")

sns.barplot(data=plot_gender, x="Gender", y="avg_order_value", color="seagreen", ax=axes[1])
axes[1].set_title("Average Order Value by Gender")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Average Order Value")

plt.tight_layout()
plt.show()

### Observation
- Male customers placed 6,780 completed orders and Female customers 6,651 (Male about 1.9% higher); 1 order has an Unknown gender.
- Revenue is almost evenly split: Male 50.1% (21.37M) and Female 49.9% (21.26M), a gap of about 0.5%.
- Female customers have a slightly higher average order value (3,196 vs 3,152, about 1.4% higher).

### Insight
Gender does not meaningfully separate high and low spenders in this dataset: Male customers order slightly more often, Female customers spend slightly more per order, and the two effects nearly cancel out. Together with the age results, customer demographics alone do not explain differences in revenue.

### Action
Avoid gender-targeted campaigns as a primary strategy. Keep marketing and pricing gender-neutral, and focus segmentation on factors that show larger differences, such as product type, loyalty membership or time period.

In [ ]:
product_summary = df_completed.groupby("Product").agg(
    units_sold=("Quantity", "sum"),
    revenue=("Total Price", "sum"),
    orders=("Total Price", "count")
).round(2)

top_units = product_summary.sort_values("units_sold", ascending=False).head(10)
top_revenue = product_summary.sort_values("revenue", ascending=False).head(10)

print("Top 10 by units sold")
display(top_units)
print("Top 10 by revenue")
display(top_revenue)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(x=top_units["units_sold"], y=top_units.index, color="steelblue", ax=axes[0])
axes[0].set_title("Top 10 Products by Units Sold")
axes[0].set_xlabel("Units Sold")
axes[0].set_ylabel("Product")

sns.barplot(x=top_revenue["revenue"] / 1_000_000, y=top_revenue.index, color="seagreen", ax=axes[1])
axes[1].set_title("Top 10 Products by Revenue")
axes[1].set_xlabel("Revenue (millions)")
axes[1].set_ylabel("Product")

plt.tight_layout()
plt.show()

### Observation
- The ten regularly sold products each sold between 7,109 and 7,612 units (about a 7% range), so unit sales barely separate them.
- Revenue differs sharply: Smartphone (SMP234) leads with 8.42M (19.7%), followed by Smartwatch (SKU1003) 6.02M (14.1%), Tablet (TBL345) 5.84M (13.7%), Smartphone (SKU1004) 5.84M (13.7%) and Laptop (LTP123) 5.06M (11.9%). These five account for about 73% of revenue.
- Tablet (SKU1002) ranks first in units (7,612) but ninth in revenue (1.88M, 4.4%) because its average unit price is 247.03. Smartphone (SMP234) sold about 3% fewer units yet earned about 4.5 times more revenue.
- Smartphone (SKU1001) sold 7,179 units but earned only 148,964 (0.3%) at an average unit price of 20.75, far below the other smartphone SKUs (791-1,140).
- Three Product Type + SKU combinations have only one order each, so they are not in the top 10.

### Insight
Because unit sales are nearly equal, revenue differences come almost entirely from price per unit, so "best-selling" depends on the definition: Tablet (SKU1002) by volume, Smartphone (SMP234) by revenue. Revenue is concentrated in higher-priced products. The very low price on Smartphone (SKU1001) is unusual and may indicate a pricing or labeling error rather than a genuinely cheap smartphone.

### Action
Use revenue as the primary measure when prioritising stock and promotions, and protect availability of the top five revenue products. Verify the price and labeling of Smartphone (SKU1001) in the source system. The business could test bundling high-volume, low-price items (such as Tablet SKU1002 and Headphones HDP456) with higher-priced products.

In [ ]:
product_summary["avg_unit_price"] = (product_summary["revenue"] / product_summary["units_sold"]).round(2)
product_summary["units_rank"] = product_summary["units_sold"].rank(ascending=False).astype(int)
product_summary["revenue_rank"] = product_summary["revenue"].rank(ascending=False).astype(int)
product_summary["revenue_share_%"] = (product_summary["revenue"] / product_summary["revenue"].sum() * 100).round(1)
product_summary.sort_values("revenue", ascending=False)

In [ ]:
category_summary = df_completed.groupby("Product Type").agg(
    revenue=("Total Price", "sum"),
    units_sold=("Quantity", "sum"),
    orders=("Total Price", "count")
).round(2)
category_summary["revenue_share_%"] = (category_summary["revenue"] / category_summary["revenue"].sum() * 100).round(1)
category_summary["avg_order_value"] = (category_summary["revenue"] / category_summary["orders"]).round(2)
category_summary = category_summary.sort_values("revenue", ascending=False)
category_summary

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=category_summary.reset_index(), x="Product Type", y="revenue", color="seagreen")
plt.title("Revenue by Product Type (Completed Orders)")
plt.xlabel("Product Type")
category_plot = category_summary.reset_index()
category_plot["revenue_millions"] = category_plot["revenue"] / 1_000_000
sns.barplot(data=category_plot, x="Product Type", y="revenue_millions", color="seagreen")
plt.ylabel("Revenue (millions)")
plt.show()

### Observation
- Smartphone is the top category with 14.41M revenue (33.8% share), followed by Smartwatch 9.40M (22.0%), Laptop 8.37M (19.6%) and Tablet 7.72M (18.1%). Headphones is lowest at 2.73M (6.4%), about one fifth of Smartphone revenue.
- The top three categories together contribute 75.4% of total revenue.
- Average order value is highest for Smartphone (3,598) and Smartwatch (3,565) and lowest for Headphones (2,009).
- Order counts follow the number of active SKUs: Smartphone has three regular SKUs (4,004 orders), Headphones has one (1,361 orders), and the others have two (2,636-2,745 orders), with each regular SKU receiving roughly 1,300-1,380 orders.

### Insight
Smartphone leads on both order volume and order value, so it is the clear revenue driver. Part of its order lead reflects having more SKUs, not only stronger customer preference. Headphones earn the least because of a lower price level and a single active SKU, which suggests room to grow rather than a proven lack of demand.

### Action
Prioritise inventory, promotion and supplier attention for Smartphone, Smartwatch and Laptop, which generate three quarters of revenue. Test expanding the Headphones range or bundling headphones with higher-priced devices to lift its share.

In [ ]:
corr_cols = ["Age", "Rating", "Unit Price", "Quantity", "Total Price", "Add-on Total"]
corr_matrix = df_completed[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Matrix of Numerical Variables (Completed Orders)")
plt.show()

### Observation
- Total Price is positively related to Unit Price (0.68) and Quantity (0.65). This is expected because Total Price is calculated as Unit Price x Quantity.
- Unit Price and Quantity are essentially unrelated (0.01).
- Rating shows a moderate negative relationship with Unit Price (-0.34) and a weaker one with Total Price (-0.23); its relationship with Quantity is about zero.
- Age is close to zero with every variable (between -0.01 and 0.00).
- Add-on Total has only weak relationships; the largest is with Unit Price (0.12).

### Insight
Order value is driven by both price and quantity, which vary independently of each other. Higher-priced items tend to receive lower ratings, a relationship that suggests price expectations may matter for satisfaction, though correlation does not show that price lowers ratings, and it may reflect differences between products. Age does not move with any spending measure, which agrees with the age-group results. Pearson correlation captures only linear relationships.

### Action
Investigate ratings of higher-priced products (for example by Product Type) to see whether satisfaction issues are concentrated there. Do not use age as a targeting variable for spending, and look to product or order characteristics for segmentation.

In [ ]:
df_clean["Is Cancelled"] = (df_clean["Order Status"] == "Cancelled").astype(int)

lost_revenue = df_clean.loc[df_clean["Is Cancelled"] == 1, "Total Price"].sum()
completed_revenue = df_completed["Total Price"].sum()
overall_rate = df_clean["Is Cancelled"].mean() * 100

print("Overall cancellation rate: ", round(overall_rate, 2), "%")
print("Revenue from cancelled orders:", round(lost_revenue, 2))
print("Revenue from completed orders:", round(completed_revenue, 2))
print("Cancelled value as % of completed revenue:", round(lost_revenue / completed_revenue * 100, 1), "%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, col in zip(axes, ["Product Type", "Payment Method", "Shipping Type"]):
    rate = (df_clean.groupby(col)["Is Cancelled"].mean() * 100).sort_values(ascending=False).round(1)
    sns.barplot(x=rate.index, y=rate.values, color="indianred", ax=ax)
    ax.axhline(overall_rate, color="black", linestyle="--", label="Overall rate")
    ax.set_title(f"Cancellation Rate by {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Cancellation Rate (%)")
    ax.tick_params(axis="x", rotation=25)
    ax.legend()
    print(col)
    print(rate, "\n")

plt.tight_layout()
plt.show()

### Observation
- 32.84% of all orders (6,568 of 20,000) were cancelled. The value of cancelled orders is 20.97M, equal to about 49.2% of completed-order revenue (42.63M).
- Cancellation rates are similar across product types (32.3%-33.1%), shipping types (32.2%-33.8%) and payment methods (30.7%-33.6%).
- Payment method shows the widest spread: Cash (30.7%) and Debit Card (31.8%) are below the overall rate, while Credit Card (33.6%) and PayPal (33.4%) are above it.

### Insight
Cancellations affect roughly one in three orders regardless of product, shipping option or payment method, which points to a business-wide process issue rather than a problem with a specific product. The payment method differences are small (under 3 percentage points) and should not be over-interpreted. Cancelled orders represent a large amount of unrealised revenue, though some customers may have re-ordered.

### Action
Investigate the order process itself (stock availability, checkout friction, payment confirmation, delivery promises) and start recording a cancellation reason so that causes can be identified. Reducing the cancellation rate even slightly would add revenue without any new customers. The payment method gap can be monitored as a secondary lead.

In [ ]:
total_orders = len(df_clean)
completed_orders = len(df_completed)
total_revenue = df_completed["Total Price"].sum()
avg_order_value = df_completed["Total Price"].mean()
median_order_value = df_completed["Total Price"].median()
unique_customers = df_completed["Customer ID"].nunique()

print("Total orders:", total_orders)
print("Completed orders:", completed_orders)
print("Completed revenue:", round(total_revenue, 2))
print("Average order value:", round(avg_order_value, 2))
print("Median order value:", round(median_order_value, 2))
print("Unique customers (completed):", unique_customers)
print("Orders per customer:", round(completed_orders / unique_customers, 2))

## Key Findings

**Overall performance**
- The dataset holds 20,000 orders; 13,432 were completed, generating 42.63M in revenue from 9,466 unique customers (about 1.42 completed orders per customer).
- Average order value is 3,173.74 against a median of 2,534.49: order values are right-skewed (skew 0.90), so a minority of large orders pulls the average up.

**Time trends**
- Monthly revenue rose from 1.31M (525 orders) in December 2023 to 4.52M (1,399 orders) in January 2024, a step driven mostly by order volume (about 2.7x) and partly by order value (about 1.3x).
- From January to August 2024 revenue stayed between 3.9M and 4.5M per month. Quarterly revenue was 12.64M (Q1), 13.23M (Q2) and 12.22M (Q3); the Q3 dip is largely due to missing days (per day: about 143.8K vs 145.4K in Q2).

**Customers**
- The Senior group (60+) has the largest revenue share (33.3%), but it also covers the widest age range (21 years); orders per year of age are similar across groups.
- Average order value is almost identical across age groups (3,143-3,194) and genders (Female 3,196, Male 3,152). Revenue is split 50.1% Male and 49.9% Female.

**Products**
- Smartphone is the top category (33.8% of revenue), followed by Smartwatch (22.0%), Laptop (19.6%) and Tablet (18.1%); Headphones is lowest (6.4%). The top three categories generate 75.4% of revenue.
- Units sold are almost equal across the ten main products (7,109-7,612), so revenue differences come from unit price. Tablet (SKU1002) ranks first in units but ninth in revenue.

**Relationships**
- Total Price is positively related to Unit Price (0.68) and Quantity (0.65), as expected from how it is calculated. Rating is moderately negatively related to Unit Price (-0.34). Age is unrelated to every variable.

**Cancellations**
- 32.84% of orders were cancelled, worth 20.97M (about 49.2% of completed revenue). Rates are similar across product type, payment method and shipping type (all between 30.7% and 33.8%).

## Business Recommendations

**1. Reduce order cancellations**
- *Finding:* 32.84% of orders are cancelled, worth 20.97M (about 49.2% of completed revenue), at similar rates across all products, payment methods and shipping types.
- *Business meaning:* The cause is likely company-wide (process or experience) rather than tied to one product or option, and a large amount of potential revenue is not being realised.
- *Recommended action:* Review the order process (stock availability, checkout, payment confirmation, delivery promises) and record a cancellation reason for every cancelled order. Set a target to lower the rate, since even a small reduction adds revenue with no new customers.

**2. Prioritise the categories that generate most revenue**
- *Finding:* Smartphone, Smartwatch and Laptop generate 75.4% of revenue; Smartphone alone contributes 33.8%.
- *Business meaning:* Customers' spending is concentrated in these categories, and stock-outs there would hit revenue hardest.
- *Recommended action:* Prioritise inventory, promotions and supplier attention for these three categories, and track availability of the top five revenue products.

**3. Use revenue, not units, to judge product performance, and grow Headphones through bundling**
- *Finding:* Units sold are nearly equal across the main products, so revenue follows price. Tablet (SKU1002) leads in units but ranks ninth in revenue, and Headphones contribute only 6.4% of revenue with a single main SKU.
- *Business meaning:* High-volume, low-price items add little revenue on their own, and Headphones have room to grow.
- *Recommended action:* Base stocking and promotion decisions on revenue contribution, and test bundling low-price items (such as Headphones and Tablet SKU1002) with higher-priced devices.

**4. Find and protect the driver behind the January 2024 step-up**
- *Finding:* Orders rose from 525 to 1,399 and average order value from 2,495 to 3,228 between December 2023 and January 2024, and both stayed stable through August 2024.
- *Business meaning:* A lasting change occurred around January 2024, but the data does not show what it was.
- *Recommended action:* Identify what changed (campaigns, launches, channels) and keep it in place. Use 2024 levels as the planning baseline and compare partial periods on a per-day basis.

**5. Segment by product and behaviour instead of age or gender**
- *Finding:* Average order value is almost identical across age groups and genders, and Age has no relationship with any spending variable.
- *Business meaning:* Age and gender do not separate high and low spenders in this data.
- *Recommended action:* Keep marketing gender-neutral and age-neutral, and build segments around product type, loyalty status or order size, where differences are larger.

**6. Investigate satisfaction on higher-priced products**
- *Finding:* Rating has a moderate negative relationship with Unit Price (-0.34).
- *Business meaning:* Higher-priced products tend to receive lower ratings, which may point to unmet expectations (correlation does not show the cause).
- *Recommended action:* Review ratings by product type for premium items and check whether complaints are concentrated in specific products.

## Conclusion

This analysis examined 20,000 electronics orders (September 2023 to September 2024) and focused on the 13,432 completed orders, which generated 42.63M in revenue from 9,466 customers. Orders are mostly modest in size (median 2,534.49), with a smaller number of large orders lifting the average to 3,173.74.

Sales stepped up sharply in January 2024, driven mainly by a near threefold rise in order volume and partly by higher order value, and then stayed stable. Customer demographics play a small role: average order value is nearly identical across age groups and genders, and age is unrelated to spending. Revenue is concentrated in products rather than people, with Smartphone, Smartwatch and Laptop generating 75.4% of revenue, while Headphones contribute only 6.4%.

The most important business issue is cancellations: about one in three orders (32.84%) is cancelled, equal to roughly half of completed revenue, at similar rates across every product, payment and shipping type. This points to a company-wide process problem and is the clearest opportunity for improvement. Together with protecting top categories, bundling low-price items and understanding the January 2024 shift, addressing it would give the business the largest gain.

**Limitations:** The dataset is synthetic, the first and last months are partial, one SKU (Smartphone SKU1001) has an unusually low unit price that should be verified, and correlation results describe relationships rather than causes.